# 통계표 노드 설계

###  0. 데이터 불러오기
- 통계청 조사 한정 통계표 목록

In [71]:
from IPython.display import display
import pandas as pd
import warnings
warnings.simplefilter(action='ignore', category=pd.errors.SettingWithCopyWarning)


path_table = "./raw_data/(참고1-2)승인통계_통계표.csv"

df_table = pd.read_csv(path_table, encoding='utf-8-sig')

In [72]:
df_table.columns

Index(['기관코드', '기관명', '승인번호', '통계명', '통계표ID', '통계표명', '수록기간(전체정보)', '주기',
       '시작연도', '마지막연도'],
      dtype='object')

In [73]:
df_table = df_table[["통계표ID", "통계표명", "승인번호", "기관코드", "주기", "시작연도", "마지막연도"]]

In [74]:
df_table

,통계표ID,통계표명,승인번호,기관코드,주기,시작연도,마지막연도
0,DT_1SA100,"세종특별자치시 농가, 농가인구",13002,101,부정기,2013,2013
1,DT_1SA101,경지규모별 농가수,13002,101,부정기,2013,2013
2,DT_1SA1011,논면적규모별 농가수,13002,101,부정기,2013,2013
3,DT_1SA1011_01,"농가수, 논면적",13002,101,부정기,2013,2013
4,DT_1SA1013,밭면적규모별 농가수,13002,101,부정기,2013,2013
...,...,...,...,...,...,...,...
11811,DT_1A02038,시도별 · 이동유형별 귀산촌인,930002,101,년,2018,2023
11812,DT_1A02039,시도별 · 성별 · 연령별 귀산촌가구주,930002,101,년,2018,2023
11813,DT_1A02040,시도별(시군별) · 가구원수별 귀산촌가구,930002,101,년,2018,2023
11814,DT_1A02041,시도별(시군별) · 가구 구성형태별 귀산촌가구,930002,101,년,2018,2023


In [75]:
df_table.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11816 entries, 0 to 11815
Data columns (total 7 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   통계표ID   11816 non-null  object
 1   통계표명    11816 non-null  object
 2   승인번호    11816 non-null  int64 
 3   기관코드    11816 non-null  int64 
 4   주기      11816 non-null  object
 5   시작연도    11816 non-null  int64 
 6   마지막연도   11816 non-null  int64 
dtypes: int64(4), object(3)
memory usage: 646.3+ KB


주기 : 5년, 4년, 3년, 2년, 년, 분기, 반기, 월, 부정기

In [76]:
df_table["주기"].value_counts()

주기
5년           5343
년            3669
2년           1577
월             333
월,분기,년        219
분기,년          213
반기            136
분기            124
부정기            90
3년             54
월,년            29
월,분기           15
반기,년            7
4년              3
분기,반기,년         2
분기,반기           1
월,분기,반기,년       1
Name: count, dtype: int64

### 1. 통계표명 간소화
- llm 활용 전 입력 토큰수 줄이기 위해
- 연령대, 지역명은 제거 후 속성으로 추가
- 시작/마지막연도가 따로 존재하므로 기간 정보는 단순 제거
- 내용과 큰 연관 없는 정보 단순 제거 (응답유형, 대분류, 전체 등)

In [77]:
df_table_name = df_table[["통계표ID","통계표명"]].copy()
df_table_name

,통계표ID,통계표명
0,DT_1SA100,"세종특별자치시 농가, 농가인구"
1,DT_1SA101,경지규모별 농가수
2,DT_1SA1011,논면적규모별 농가수
3,DT_1SA1011_01,"농가수, 논면적"
4,DT_1SA1013,밭면적규모별 농가수
...,...,...
11811,DT_1A02038,시도별 · 이동유형별 귀산촌인
11812,DT_1A02039,시도별 · 성별 · 연령별 귀산촌가구주
11813,DT_1A02040,시도별(시군별) · 가구원수별 귀산촌가구
11814,DT_1A02041,시도별(시군별) · 가구 구성형태별 귀산촌가구


#### 1) 단어 빈도수 확인

전체 확인

In [78]:
import pandas as pd
from collections import Counter
import re


# 모든 통계표명 합치기 (소문자 통일 + 특수문자 제거)
all_text = " ".join(df_table["통계표명"].astype(str))
all_text = re.sub(r"[^\w\s]", " ", all_text)  # 특수문자 제거
all_text = all_text.lower()

# 띄어쓰기 기준 단어 분리
tokens = all_text.split()

# 단어 빈도수 계산
word_counts = Counter(tokens)

# 상위 N개 단어 보기
top_n = 1000
for word, count in word_counts.most_common(top_n):
    print(f"{word}: {count}")


및: 2502
시도: 1976
이상: 1264
성별: 1141
인구: 1059
산업: 972
성: 958
농가: 672
연령별: 669
연령: 649
가구: 538
사업체수: 514
시군구: 477
종사자수: 467
규모별: 465
총괄: 413
면적: 406
산업별: 397
일반가구: 394
15세: 388
취업자: 379
15세이상: 346
행정구역: 337
현황: 314
가구당: 308
월평균: 307
어가: 287
13세: 286
종사자규모별: 282
이동: 276
일자리: 268
조직형태별: 268
가구주: 267
교육정도별: 256
대한: 251
가계수지: 251
20세이상: 250
개인농가: 244
수: 243
거처의: 232
종사상지위별: 228
행위자비율: 227
직업: 218
시도별: 215
가구주의: 208
별: 200
전국: 194
혼인상태별: 182
복수응답: 178
2인이상: 174
직업별: 173
임가: 168
전: 163
가구원: 162
거주지별: 159
종류: 157
가구원수별: 143
현거주지: 142
만족도: 141
93: 140
평균시간: 139
시: 138
이민자: 138
매출액: 137
1인가구: 133
행동비율: 133
농가수: 132
주된: 131
현: 131
수확면적: 130
어업: 130
신혼부부: 129
20: 124
행위자평균시간: 124
교육정도: 123
고령자: 123
해수면어업: 123
산업중분류별: 119
종류별: 115
19세: 115
1인이상: 113
산업세세분류별: 110
경영주: 110
재배면적: 109
기업체수: 108
주요지표: 105
주된응답: 105
어가인구: 105
혼인상태: 104
주거전용: 104
통근: 103
가구내구재: 102
취업자의: 101
세대구성: 99
10인: 97
소득: 96
점유형태별: 95
가구수: 93
도시: 92
연간소득: 92
인구의: 92
이유: 90
지난: 90
06: 90
종사자: 89
농가인구: 88
중분류: 88
자산: 88
경제활동상태별: 87
유형별

괄호 내용 확인

In [79]:
import re
from collections import Counter

# 전체 텍스트 결합 (통계표명 컬럼 전체)
all_text = " ".join(df_table["통계표명"].astype(str))

# 괄호 안 단어 추출: (단어)
bracket_words = re.findall(r"\((.*?)\)", all_text)

# 빈도수 계산
bracket_counts = Counter(bracket_words)

# 상위 N개 출력
top_n = 100
for word, count in bracket_counts.most_common(top_n):
    print(f"{word}: {count}")


개인농가: 244
일반가구: 243
15세 이상: 215
15세이상: 213
시도: 172
해수면어업: 123
’20~ : 120
이민자: 99
2인이상 가구: 96
10인 이상: 92
중분류: 84
`93~`05: 84
농가경제: 78
13세 이상 인구: 76
대분류: 74
내수면어업: 73
9개도: 72
`06~  : 67
일반가구, 주거전용: 63
12세 이상: 60
'93-'98: 56
농림어업제외: 55
총농가: 55
60세 이상: 52
20세이상: 50
시계열 보정 前 자료: 46
1인가구: 46
19세 이상: 45
5세 이상: 44
복수응답: 44
전국,1인이상: 42
도시,1인이상: 42
주거전용: 40
가구주: 39
20세이상 인구: 38
시도/시/군/구: 37
소분류: 33
영주(F-5: 30
5세: 28
주된응답: 28
12세이상: 26
20세이상, 일반가구: 26
전국,2인이상: 26
도시,2인이상: 26
19세 이상 인구: 26
10세 이상: 26
2020=100: 25
근무지기준: 24
6세이상: 23
10차: 22
15세 이상 취업자: 22
65세 이상: 21
세세분류: 21
주된응답, 13세 이상 인구: 21
복수응답, 13세 이상 인구: 21
60세이상: 20
활동/신생/소멸: 20
명목, 연말기준: 20
F-4: 19
2009년: 18
읍면동/성/연령별: 17
`06~   : 17
세분류: 16
5명이상: 16
지난 1년간, 복수응답, 13세 이상 인구: 16
준농가: 16
13세 이상: 15
15세이상인구: 15
10세: 15
6세 이상: 14
9차: 14
전국,1인이상,실질: 14
도시,1인이상,실질: 14
시군구: 14
명목: 14
지난 1년간, 이민자: 14
1세 이상: 13
20세 이상 인구: 13
65세이상: 12
14세이상: 12
12세이하, 일반가구: 12
고졸이상: 12
변동전: 12
비전문취업: 12
시군별: 12
구시군: 11
5세이상: 11
9차, ~2017: 11
15세이상 인구: 11
중복응답: 11
봉

#### 2) 지역명 제거

In [80]:
df_txt = pd.read_csv("./raw_data/법정동코드 전체자료.txt", sep="\t", encoding="cp949")

region_set = set()

for name in df_txt["법정동명"].values:
    parts = name.strip().split()
    if len(parts) == 1:
        region_set.add(parts[0])
    elif len(parts) >= 2:
        if parts[0].endswith("시"):
            region_set.add(parts[0])
        else:
            region_set.add(parts[0])
            region_set.add(parts[1])


manual_regions = [
    "전국", "서울", "인천", "경기", "대구", "경북", "광주",
    "전남", "대전", "충북", "충남", "강원", "전북", "제주"
]

# 정규식 패턴 생성
left_boundary = r"(^|[ ,()·])"
right_boundary = r"(?=[ ,()·]|$)"

region_pattern1 = left_boundary + "(" + "|".join(sorted(region_set, key=len, reverse=True)) + ")" + right_boundary
region_pattern2 = left_boundary + "(" + "|".join(map(re.escape, manual_regions)) + ")" + right_boundary



# 4. 함수 정의: 두 패턴 모두에서 지역 추출 및 제거
def extract_regions_sequential(text):
    found_regions = set()

    # pattern1 적용
    matches1 = re.findall(region_pattern1, text)
    for match in matches1:
        found_regions.add(match[1])  # 두 번째 그룹이 실제 지역명
        text = re.sub(re.escape(match[1]), "", text)

    # pattern2 적용 (pattern1 제거 후 텍스트에)
    matches2 = re.findall(region_pattern2, text)
    for match in matches2:
        found_regions.add(match[1])
        text = re.sub(re.escape(match[1]), "", text)

    return pd.Series([", ".join(sorted(found_regions)), text.strip()])

# 적용
df_table_name[["지역", "cleaning1"]] = df_table_name["통계표명"].astype(str).apply(extract_regions_sequential)


In [81]:
import re

def clean_region_name(region):
    if pd.isna(region):
        return region

    # 1. 광역시, 특별자치시, 특별자치도 삭제
    region = re.sub(r"(광역시|특별자치시|특별자치도)", "", region)

    # 2. 충청남도 → 충남, 충청북도 → 충북 등 축약
    region = region.replace("충청남도", "충남").replace("충청북도", "충북")
    region = region.replace("경상남도", "경남").replace("경상북도", "경북")
    region = region.replace("전라남도", "전남").replace("전라북도", "전북")
    region = region.replace("강원도", "강원").replace("제주도", "제주")

    # 3. 앞뒤 공백 제거
    return region.strip()

# 적용
df_table_name["지역"] = df_table_name["지역"].astype(str).apply(clean_region_name)


In [82]:
df_filtered = df_table_name[df_table_name["지역"].str.strip() != ""]
df_filtered

,통계표ID,통계표명,지역,cleaning1
0,DT_1SA100,"세종특별자치시 농가, 농가인구",세종,"농가, 농가인구"
90,DT_1B9000D,"현거주지/성/목적지별 통근·통학인구(서울,인천,경기,12세이상)","경기, 서울, 인천","현거주지/성/목적지별 통근·통학인구(,,,12세이상)"
92,DT_1B9000F,"현거주지/성/목적지별 통근·통학인구(대구,경북,12세이상)","경북, 대구","현거주지/성/목적지별 통근·통학인구(,,12세이상)"
93,DT_1B9000G,"현거주지/성/목적지별 통근·통학인구(광주,전남,12세이상)","광주, 전남","현거주지/성/목적지별 통근·통학인구(,,12세이상)"
94,DT_1B9000H,"현거주지/성/목적지별 통근·통학인구(대전,충남,12세이상)","대전, 충남","현거주지/성/목적지별 통근·통학인구(,,12세이상)"
...,...,...,...,...
11013,INH_1K52B06_39,제주특별자치도 · 산업 · 종사자성별 종사자수(`06~ ),제주,· 산업 · 종사자성별 종사자수(`06~ )
11014,INH_1K52C04_39,제주특별자치도·산업·종사상지위별 종사자수,제주,·산업·종사상지위별 종사자수
11015,INH_1K52C05_39,제주특별자치도·산업·대표자성별 사업체수,제주,·산업·대표자성별 사업체수
11016,INH_1K52C06_39,제주특별자치도·산업·종사자성별 사업체수,제주,·산업·종사자성별 사업체수


#### 3) 수록기간 제거

In [83]:
import re
from collections import Counter

period_pattern = r"\(['’`]\d{2}[~-][^)]*\)"

all_text = " ".join(df_table["통계표명"].astype(str))
matches = re.findall(period_pattern, all_text)

counted = Counter(matches)

for bracket_str, cnt in counted.most_common():
    print(f"{bracket_str}: {cnt}회")

(’20~ ): 120회
(`93~`05): 84회
(`06~  ): 67회
('93-'98): 56회
(`06~   ): 17회
(’06~ ): 6회
(’16~ ): 2회
('12~   ): 1회
(’12~ ): 1회


In [84]:
def remove_pattern(text):
    return re.sub(period_pattern, "", text)

df_table_name["cleaning2"] = df_table_name["cleaning1"].astype(str).apply(remove_pattern)

changed_rows = df_table_name[df_table_name["cleaning2"].astype(str) != df_table_name["cleaning1"].astype(str)]

# 결과 출력
changed_rows[["통계표명","cleaning1","cleaning2"]]

,통계표명,cleaning1,cleaning2
5739,시도별/구산업별/사업체구분별('93-'98),시도별/구산업별/사업체구분별('93-'98),시도별/구산업별/사업체구분별
5740,시도별/구산업별/조직형태별('93-'98),시도별/구산업별/조직형태별('93-'98),시도별/구산업별/조직형태별
5741,시도별/구산업별/종사자규모별('93-'98),시도별/구산업별/종사자규모별('93-'98),시도별/구산업별/종사자규모별
5742,시도별/구산업별/종사상지위별('93-'98),시도별/구산업별/종사상지위별('93-'98),시도별/구산업별/종사상지위별
5743,"시도/산업/사업체구분별 사업체수, 종사자수(`93~`05)","시도/산업/사업체구분별 사업체수, 종사자수(`93~`05)","시도/산업/사업체구분별 사업체수, 종사자수"
...,...,...,...
11009,"제주특별자치도 · 산업 · 조직형태별 사업체수, 종사자수(`06~ )","· 산업 · 조직형태별 사업체수, 종사자수(`06~ )","· 산업 · 조직형태별 사업체수, 종사자수"
11010,"제주특별자치도 · 산업 · 종사자규모별 사업체수, 종사자수(`06~ )","· 산업 · 종사자규모별 사업체수, 종사자수(`06~ )","· 산업 · 종사자규모별 사업체수, 종사자수"
11011,"제주특별자치도 · 산업 · 종사자지위별, 종사자수(`06~ )","· 산업 · 종사자지위별, 종사자수(`06~ )","· 산업 · 종사자지위별, 종사자수"
11012,제주특별자치도 · 산업 · 대표자성별 사업체수(`06~ ),· 산업 · 대표자성별 사업체수(`06~ ),· 산업 · 대표자성별 사업체수


#### 4) 연령대 제거

In [85]:
import re
from collections import Counter

age_pattern = r"\d{1,3}세\s?(?:이상|이하|미만|초과)"

text = " ".join(df_table["통계표명"].astype(str))
matches = re.findall(age_pattern, text)
full_bases = {re.match(r"\d{1,3}세", m).group() for m in matches}
counted_full = Counter(matches)

# 출력
for k, v in counted_full.most_common():
    print(f"{k}: {v}회")

15세 이상: 387회
15세이상: 368회
13세 이상: 286회
20세이상: 276회
19세 이상: 115회
60세 이상: 79회
12세 이상: 63회
10세이상: 54회
5세 이상: 51회
12세이상: 48회
60세이상: 35회
14세이상: 33회
6세이상: 30회
10세 이상: 26회
65세 이상: 24회
20세 이상: 23회
6세 이상: 22회
1세 이상: 16회
19세이상: 14회
65세이상: 12회
12세이하: 12회
30세 이상: 12회
5세이상: 11회
14세 이상: 8회
1세이상: 7회
12세 이하: 6회
10세이하: 5회
18세 미만: 5회
18세이상: 4회
30세이상: 4회
13세이상: 3회
7세이상: 3회
0세 이상: 3회
18세 이하: 2회
18세 이상: 2회
2세이하: 1회
24세 이하: 1회
0세이상: 1회
40세 이상: 1회


In [86]:
sum(counted_full.values())

2053

In [87]:
import re
import pandas as pd

pattern_full = r"\d{1,3}세\s?(?:이상|이하|미만|초과)"
# pattern_simple = r"[ ,()·]\d{1,3}세"

def extract_and_clean_age(text):
    found_ages = []

    # 1. pattern_full 찾기
    matches_full = re.findall(pattern_full, text)
    found_ages.extend(matches_full)

    # 1-1. pattern_full 제거
    cleaned = text
    for m in matches_full:
        cleaned = re.sub(re.escape(m), "", cleaned)

    # 2. pattern_simple 찾기 (pattern_full 제거된 텍스트에서)
    # matches_simple = re.findall(pattern_simple, cleaned)
    # found_ages.extend(matches_simple)

    # 2-1. pattern_simple 제거
    # for m in matches_simple:
    #     cleaned = re.sub(r"\b" + re.escape(m) + r"\b", "", cleaned)

    # 연령대는 중복 제거 후 쉼표로 연결
    found_ages = sorted(set(found_ages))

    return pd.Series([", ".join(found_ages), cleaned.strip()])

# 적용
df_table_name[["연령대", "cleaning3"]] = df_table_name["cleaning2"].astype(str).apply(extract_and_clean_age)

In [88]:
# 빈 괄호 또는 괄호 안에 문자열이 하나만 있는 경우 삭제

def remove_parentheses(text):
    return re.sub(r"\([^)]{0,1}\)", "", text)

df_table_name["cleaning3"] = df_table_name["cleaning3"].astype(str).apply(remove_parentheses)

In [89]:
# 결과 확인
df_filtered = df_table_name[df_table_name["연령대"].str.strip() != ""]
df_filtered[["통계표명", "연령대", "cleaning3"]]

,통계표명,연령대,cleaning3
66,성별/연령별/혼인상태별 1인가구수(15세이상),15세이상,성별/연령별/혼인상태별 1인가구수
69,성별/연령별/교육정도별 인구(6세이상),6세이상,성별/연령별/교육정도별 인구
70,성별/연령별/혼인상태별 인구(15세이상),15세이상,성별/연령별/혼인상태별 인구
83,행정구역/가구규모/성별 통근·통학인구(12세이상),12세이상,행정구역/가구규모/성별 통근·통학인구
84,대도시/통근통학/성/이용교통수단별 통근·통학인구(12세이상),12세이상,대도시/통근통학/성/이용교통수단별 통근·통학인구
...,...,...,...
9841,"녹색생활 실천하는 이유(녹색생활 실천하는 사람, 20세이상 인구)",20세이상,"녹색생활 실천하는 이유(녹색생활 실천하는 사람, 인구)"
9842,"녹색생활 실천하지 않는 이유(녹색생활 실천하지 않는 사람, 20세이상 인구)",20세이상,"녹색생활 실천하지 않는 이유(녹색생활 실천하지 않는 사람, 인구)"
11347,15세 이상 국내 상주인구(이민자),15세 이상,국내 상주인구(이민자)
11733,개인특성별 경제활동상태별 15세이상 인구,15세이상,개인특성별 경제활동상태별 인구


#### 5) 기타
- 응답유형 제거 : 주된응답, 복수응답 등
- 분류유형 제거 : 소분류, 중분류, 대·중분류 등
- 포괄적인 단어 제거 : 전체, 전국

In [90]:
import re

pattern = (
    r"(주된\s?응답|복수\s?응답|단일\s?응답|다중\s?응답|미응답|무응답),?\s?"  # 응답 유형 제거
    r"|" 
    r"\([^\s)]{1,3}분류\)"                       # 괄호 안 글자 최대 3자 + 분류 제거
    r"|" 
    r"(\(전체\),|-\s?전체|\(전국\),|-\s?전국)"     # 전체/전국 단어 제거
)

def remove_terms(text):
    return re.sub(pattern, "", text)

df_table_name["cleaning4"] = df_table_name["cleaning3"].astype(str).apply(remove_terms)


In [91]:
# 괄호 제거
df_table_name["cleaning4"] = df_table_name["cleaning4"].astype(str).apply(remove_parentheses)

In [92]:
# 결과 출력
changed_rows = df_table_name[df_table_name["cleaning4"].astype(str) != df_table_name["cleaning3"].astype(str)]

display(changed_rows[["통계표ID","cleaning3","cleaning4"]])

,통계표ID,cleaning3,cleaning4
147,DT_1BAOO03,시도/성/직업(대분류)/산업(대분류)별 취업자,시도/성/직업/산업별 취업자
148,DT_1BAOO04,행정구역/성/연령/산업(대분류)별 취업자,행정구역/성/연령/산업별 취업자
149,DT_1BAOO05,행정구역/성/연령/직업(대분류)별 취업자,행정구역/성/연령/직업별 취업자
152,DT_1BAOO08,"시도/성/연령,혼인상태/산업(대분류)별 취업자","시도/성/연령,혼인상태/산업별 취업자"
153,DT_1BAOO09,"시도/성/연령,혼인상태/직업(대분류)별 취업자","시도/성/연령,혼인상태/직업별 취업자"
...,...,...,...
11441,DT_2FP008R,"한국생활에서 어려운 사항(복수응답, 이민자)",한국생활에서 어려운 사항(이민자)
11443,DT_2FP010R,"여가 활용 형태(복수응답, 이민자)",여가 활용 형태(이민자)
11455,DT_2FQ002R,"한국어 학습 경험 유무 및 학습기관(복수응답, 이민자)",한국어 학습 경험 유무 및 학습기관(이민자)
11471,DT_2FR007R,"취업할 때 일자리 정보 획득 경로(복수응답, 방문취업(H-2), 재외동포(F-4) ...","취업할 때 일자리 정보 획득 경로(방문취업(H-2), 재외동포(F-4) 한국계외국인)"


새로운 속성 "지역", "연령대"를 기존 df에 적용

In [93]:
df_table["지역"]=df_table_name["지역"]
df_table["연령대"]=df_table_name["연령대"]

df_table

,통계표ID,통계표명,승인번호,기관코드,주기,시작연도,마지막연도,지역,연령대
0,DT_1SA100,"세종특별자치시 농가, 농가인구",13002,101,부정기,2013,2013,세종,
1,DT_1SA101,경지규모별 농가수,13002,101,부정기,2013,2013,,
2,DT_1SA1011,논면적규모별 농가수,13002,101,부정기,2013,2013,,
3,DT_1SA1011_01,"농가수, 논면적",13002,101,부정기,2013,2013,,
4,DT_1SA1013,밭면적규모별 농가수,13002,101,부정기,2013,2013,,
...,...,...,...,...,...,...,...,...,...
11811,DT_1A02038,시도별 · 이동유형별 귀산촌인,930002,101,년,2018,2023,,
11812,DT_1A02039,시도별 · 성별 · 연령별 귀산촌가구주,930002,101,년,2018,2023,,
11813,DT_1A02040,시도별(시군별) · 가구원수별 귀산촌가구,930002,101,년,2018,2023,,
11814,DT_1A02041,시도별(시군별) · 가구 구성형태별 귀산촌가구,930002,101,년,2018,2023,,


### 2. 구분 및 대상 추출
- ex) 산업별 취업자 -> 구분: 산업, 대상: 취업자

In [94]:
# 상위 빈도 단어 중에서 '별'로 끝나는 단어만 필터링
top_n = 150  # 필요 시 조절
별로_끝나는_단어 = [(word, count) for word, count in word_counts.most_common(top_n) if word.endswith("별")]

# 출력
for word, count in 별로_끝나는_단어:
    print(f"{word}: {count}")


성별: 1141
연령별: 669
규모별: 465
산업별: 397
종사자규모별: 282
조직형태별: 268
교육정도별: 256
종사상지위별: 228
시도별: 215
별: 200
혼인상태별: 182
직업별: 173
거주지별: 159
가구원수별: 143
산업중분류별: 119
종류별: 115
산업세세분류별: 110
점유형태별: 95
경제활동상태별: 87
유형별: 87
도별: 85
매출액규모별: 83
지역별: 82
수확면적규모별: 81
특성별: 79
산업대분류별: 75
자본금규모별: 72
사업체구분별: 72
재배면적규모별: 72
세대구성별: 71
취업여부별: 70
사육규모별: 69


규모별의 경우 종류가 다양하였음 : 종사자규모별, 매출액규모별, 수화면적규모별, 재배면적규모별, 사육규모별 ...

In [95]:
# "규모별" 포함 단어들 총합 계산
규모별_포함_총합 = sum(count for word, count in 별로_끝나는_단어 if "규모별" in word)
포함_단어들 = [(word, count) for word, count in 별로_끝나는_단어 if "규모별" in word]

# "규모별" 미포함 단어들의 개수와 총합 계산
나머지_단어들 = [(word, count) for word, count in 별로_끝나는_단어 if "규모별" not in word]
나머지_개수 = len(나머지_단어들)
나머지_총합 = sum(count for _, count in 나머지_단어들)

print(f'"규모별" 포함 단어 총합: {규모별_포함_총합}회')
print(f'"규모별" 미포함 단어 개수: {나머지_개수}개')
print(f'"규모별" 미포함 단어 총합: {나머지_총합}회')

print("\n[규모별 포함 단어 목록]")
for word, count in 포함_단어들:
    print(f"{word}: {count}회")

"규모별" 포함 단어 총합: 1124회
"규모별" 미포함 단어 개수: 25개
"규모별" 미포함 단어 총합: 5178회

[규모별 포함 단어 목록]
규모별: 465회
종사자규모별: 282회
매출액규모별: 83회
수확면적규모별: 81회
자본금규모별: 72회
재배면적규모별: 72회
사육규모별: 69회


In [96]:
df_table

,통계표ID,통계표명,승인번호,기관코드,주기,시작연도,마지막연도,지역,연령대
0,DT_1SA100,"세종특별자치시 농가, 농가인구",13002,101,부정기,2013,2013,세종,
1,DT_1SA101,경지규모별 농가수,13002,101,부정기,2013,2013,,
2,DT_1SA1011,논면적규모별 농가수,13002,101,부정기,2013,2013,,
3,DT_1SA1011_01,"농가수, 논면적",13002,101,부정기,2013,2013,,
4,DT_1SA1013,밭면적규모별 농가수,13002,101,부정기,2013,2013,,
...,...,...,...,...,...,...,...,...,...
11811,DT_1A02038,시도별 · 이동유형별 귀산촌인,930002,101,년,2018,2023,,
11812,DT_1A02039,시도별 · 성별 · 연령별 귀산촌가구주,930002,101,년,2018,2023,,
11813,DT_1A02040,시도별(시군별) · 가구원수별 귀산촌가구,930002,101,년,2018,2023,,
11814,DT_1A02041,시도별(시군별) · 가구 구성형태별 귀산촌가구,930002,101,년,2018,2023,,


In [97]:
df_table.to_csv("./nodes/node_table.csv", index=False, encoding="utf-8-sig")

#### [문제점 및 계획]

- 정규식으로 제거하기 까다로움 

    "-별"로 반드시 끝나지 않고 여러개의 구분기준들이 특수기호로 이어진 경우가 많았음
- 주요 구분 단어

    keywords = ["성별","연령별","산업별","시도별","조직형태별","교육정도별","행정구역별","종사상지위별",
            "혼인상태별","직업별","거주지별","가구원수별","점유형태별","경제활동상태별","매출액규모별"]

- To do list
1. df_table_name["cleaning4"]의 각 데이터를 입력값으로 하여 "구분", "대상"을 출력값으로 하는 프롬프트 템플릿 제작
2. 템플릿을 config.yaml에 저장
3. llm 적용하여 잘 실행되는지 확인
4. "구분", "대상"을 df_table_name의 새로운 속성으로 만들기

In [98]:
df_table_name[["통계표ID","통계표명","cleaning4"]]

,통계표ID,통계표명,cleaning4
0,DT_1SA100,"세종특별자치시 농가, 농가인구","농가, 농가인구"
1,DT_1SA101,경지규모별 농가수,경지규모별 농가수
2,DT_1SA1011,논면적규모별 농가수,논면적규모별 농가수
3,DT_1SA1011_01,"농가수, 논면적","농가수, 논면적"
4,DT_1SA1013,밭면적규모별 농가수,밭면적규모별 농가수
...,...,...,...
11811,DT_1A02038,시도별 · 이동유형별 귀산촌인,시도별 · 이동유형별 귀산촌인
11812,DT_1A02039,시도별 · 성별 · 연령별 귀산촌가구주,시도별 · 성별 · 연령별 귀산촌가구주
11813,DT_1A02040,시도별(시군별) · 가구원수별 귀산촌가구,시도별(시군별) · 가구원수별 귀산촌가구
11814,DT_1A02041,시도별(시군별) · 가구 구성형태별 귀산촌가구,시도별(시군별) · 가구 구성형태별 귀산촌가구


## 구분/대상 추출

In [169]:
import os
import yaml
import time
from dotenv import load_dotenv
from openai import OpenAI


# 🔹 CLOVA API, URL 로드
load_dotenv()
CLOVA_API_KEY = os.getenv("CLOVASTUDIO_API_KEY")
BASE_URL = os.getenv("CLOVASTUDIO_API_BASE_URL")


# 🔹 프롬프트 템플릿 로드
with open("config.yaml", "r", encoding="utf-8") as f:
    config = yaml.load(f, Loader=yaml.FullLoader)
PROMPT_TEMPLATE_1 = config["criteria_extraction_template"]
PROMPT_TEMPLATE_2 = config["target_extraction_template"]

# 🔹 OpenAI 호환 client 구성
client = OpenAI(
    api_key=CLOVA_API_KEY,
    base_url=BASE_URL
)
def execute_clovastudio(user_input: str, prompt_template: str) -> str:
    prompt = prompt_template.format(table_name=user_input)

    response = client.chat.completions.create(
        model="HCX-005",
        messages=[
            {"role": "user", "content": prompt}
        ]
    )
    # time.sleep(0.1)
    return response.choices[0].message.content

In [ ]:
# test

user_input = df_table_name["cleaning4"].values[11814]
criteria = execute_clovastudio(user_input, PROMPT_TEMPLATE_1)
target = execute_clovastudio(user_input, PROMPT_TEMPLATE_2)

print('[input]:',user_input)
print('[criteria]:',criteria)
print('[target]:',target)

[input]: 시도별(시군별) · 가구 구성형태별 귀산촌가구
[criteria]: 행정구역별, 가구 구성형태별
[target]: 귀산촌가구


### Clova Studio API 호출 속도 최적화
- 429 Too Many Requests 에러가 반복적으로 발생
- QPS 제한(초당 요청 수 제한)이 있음
- 요청 60건마다 time.sleep()을 사용해 속도 제한을 인위적으로 적용
- 경과 시간(초)이 호출 수(idx)보다 빠르면 차이값 만큼 대기.
- 요청 간 평균 처리 속도를 1초당 1건으로 유지
- cf) 60 QPM(분당 요청 수)은 1 QPS를 의미

In [173]:
import time
from tqdm import tqdm

results = []
start_time = time.time() - 30

for idx, text in enumerate(tqdm(df_table_name["cleaning4"], desc="Processing rows")):
    retry_count = 0

    if idx == 0:
        time.sleep(30)

    while True:
        try:
            if (idx) % 60 == 0:
                elapsed = int(time.time() - start_time + 1)
                wait_time = idx - elapsed + 1
                # print(f"idx {idx}, gap {elapsed}, start {start_time}, now {time.time()}")
                if wait_time > 0:
                    time.sleep(wait_time + 1)
                else:
                    pass
            result = execute_clovastudio(text, PROMPT_TEMPLATE_1)
            break  # 성공 시 루프 탈출
        except Exception as e:            
            retry_count += 1
            elapsed = int(time.time() - start_time + 1)
            wait_time = idx - elapsed + 1
            if wait_time > 0:
                time.sleep(wait_time + 1)
            if e == 429 :
                print(f"Error at index {idx}: {e}- Waiting {wait_time} seconds before retrying... (attempt {retry_count})")


    results.append(result)



df_table_name["구분"] = results


Processing rows: 100%|██████████| 11816/11816 [3:18:13<00:00,  1.01s/it]  


In [175]:
import time
from tqdm import tqdm

results = []
start_time = time.time() - 30

for idx, text in enumerate(tqdm(df_table_name["cleaning4"], desc="Processing rows")):
    retry_count = 0

    if idx == 0:
        time.sleep(30)

    while True:
        try:
            if (idx) % 60 == 0:
                elapsed = int(time.time() - start_time + 1)
                wait_time = idx - elapsed + 1
                # print(f"idx {idx}, gap {elapsed}, start {start_time}, now {time.time()}")
                if wait_time > 0:
                    time.sleep(wait_time + 1)
                else:
                    pass
            result = execute_clovastudio(text, PROMPT_TEMPLATE_2)
            break  # 성공 시 루프 탈출
        except Exception as e:            
            retry_count += 1
            elapsed = int(time.time() - start_time + 1)
            wait_time = idx - elapsed + 1
            if wait_time > 0:
                time.sleep(wait_time + 1)
            if e == 429 :
                print(f"Error at index {idx}: {e}- Waiting {wait_time} seconds before retrying... (attempt {retry_count})")


    results.append(result)



df_table_name["대상"] = results

Processing rows: 100%|██████████| 11816/11816 [3:20:17<00:00,  1.02s/it]  


In [178]:
df_table_name

,통계표ID,통계표명,지역,cleaning1,cleaning2,연령대,cleaning3,cleaning4,구분,대상
0,DT_1SA100,"세종특별자치시 농가, 농가인구",세종,"농가, 농가인구","농가, 농가인구",,"농가, 농가인구","농가, 농가인구",기준없음,"농가, 농가인구"
1,DT_1SA101,경지규모별 농가수,,경지규모별 농가수,경지규모별 농가수,,경지규모별 농가수,경지규모별 농가수,경지규모별,농가수
2,DT_1SA1011,논면적규모별 농가수,,논면적규모별 농가수,논면적규모별 농가수,,논면적규모별 농가수,논면적규모별 농가수,논면적 규모별,농가수
3,DT_1SA1011_01,"농가수, 논면적",,"농가수, 논면적","농가수, 논면적",,"농가수, 논면적","농가수, 논면적",기준없음,"농가수, 논면적"
4,DT_1SA1013,밭면적규모별 농가수,,밭면적규모별 농가수,밭면적규모별 농가수,,밭면적규모별 농가수,밭면적규모별 농가수,밭면적규모별,농가수
...,...,...,...,...,...,...,...,...,...,...
11811,DT_1A02038,시도별 · 이동유형별 귀산촌인,,시도별 · 이동유형별 귀산촌인,시도별 · 이동유형별 귀산촌인,,시도별 · 이동유형별 귀산촌인,시도별 · 이동유형별 귀산촌인,"행정구역별, 이동유형별",귀산촌인
11812,DT_1A02039,시도별 · 성별 · 연령별 귀산촌가구주,,시도별 · 성별 · 연령별 귀산촌가구주,시도별 · 성별 · 연령별 귀산촌가구주,,시도별 · 성별 · 연령별 귀산촌가구주,시도별 · 성별 · 연령별 귀산촌가구주,"시도별, 성별, 연령별",귀산촌가구주
11813,DT_1A02040,시도별(시군별) · 가구원수별 귀산촌가구,,시도별(시군별) · 가구원수별 귀산촌가구,시도별(시군별) · 가구원수별 귀산촌가구,,시도별(시군별) · 가구원수별 귀산촌가구,시도별(시군별) · 가구원수별 귀산촌가구,"행정구역별, 가구원수별",귀산촌가구
11814,DT_1A02041,시도별(시군별) · 가구 구성형태별 귀산촌가구,,시도별(시군별) · 가구 구성형태별 귀산촌가구,시도별(시군별) · 가구 구성형태별 귀산촌가구,,시도별(시군별) · 가구 구성형태별 귀산촌가구,시도별(시군별) · 가구 구성형태별 귀산촌가구,"행정구역별, 가구 구성형태별",귀산촌가구


In [177]:
df_table_name.to_csv("./nodes/table_total.csv", index=False, encoding="utf-8-sig")

In [ ]:
import pandas as pd
from gensim.models import KeyedVectors

# fastText 벡터 로딩 (압축된 경우 자동 해제됨)
fasttext_model = KeyedVectors.load_word2vec_format('cc.ko.300.vec.gz', binary=False)

In [ ]:
# df_table_name["대상"]에 쉼표로 나뉜 단어들이 있다고 가정
target_list = set()

for items in df_table_name["대상"].dropna():
    for word in items.split(","):
        target_list.add(word.strip())

target_list = list(target_list)

In [ ]:
from itertools import combinations

edges = []

for w1, w2 in combinations(target_list, 2):
    if w1 in fasttext_model.key_to_index and w2 in fasttext_model.key_to_index:
        similarity = fasttext_model.similarity(w1, w2)
        # 유사도 임계값 설정 (예: 0.7 이상만 관계 생성)
        if similarity >= 0.7:
            edges.append((w1, w2, round(similarity, 4)))

In [ ]:
edges_target = pd.DataFrame(edges, columns=["source", "target", "similarity"])
edges_target.to_csv("target_similarity.csv", index=False)